In [ ]:
pip install --upgrade pip setuptools wheel


In [ ]:
# Step 1: Install Required Libraries
!pip install -q transformers accelerate datasets peft bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 123.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 143.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 173.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 138.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 131.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 123.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 98.3 MB/s eta 0:00:00


In [ ]:
# Step 2: Import Libraries
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import os
import json
import shutil
from google.colab import files
from huggingface_hub import notebook_login

In [ ]:
# Authenticate Hugging Face
notebook_login()

In [ ]:
# Step 3: Load the Tokenizer & Model
model_name = "xlm-roberta-base"

# ✅ Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ✅ Load Pretrained XLM-R Model for Classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2, device_map="auto"
)

print("✅ XLM-R Model Loaded Successfully!")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ XLM-R Model Loaded Successfully!


In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# Step 4: Load & Preprocess Dataset from Your Device
from google.colab import files

# Upload the file
uploaded = files.upload()

# Get the filename (assumes only one file is uploaded)
file_name = list(uploaded.keys())[0]

# ✅ Load the dataset
df = pd.read_csv(file_name)

# Ensure column names match expectations
df.rename(columns={"Sentiment": "Label", "Reviews": "Review"}, inplace=True)

# Convert Label values: 0 -> positive, 1 -> negative (if needed)
df["Label"] = df["Label"].astype(int)  # Ensure labels are integers

# ✅ Split into train and test sets
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["Review"], df["Label"], test_size=0.2, random_state=42
)

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(pd.DataFrame({"Review": train_texts, "Label": train_labels}))
test_dataset = Dataset.from_pandas(pd.DataFrame({"Review": test_texts, "Label": test_labels}))

dataset = DatasetDict({"train": train_dataset, "test": test_dataset})

print("✅ Dataset Loaded and Split!")

# ✅ Tokenization Function
def tokenize_function(examples):
    tokens = tokenizer(examples["Review"], padding="max_length", truncation=True, max_length=256)
    tokens["labels"] = examples["Label"]
    return tokens

# Apply Tokenization
tokenized_train = dataset["train"].map(tokenize_function, batched=True)
tokenized_test = dataset["test"].map(tokenize_function, batched=True)

# Convert to Torch format
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

tokenized_datasets = {"train": tokenized_train, "test": tokenized_test}

print("✅ Tokenization Completed!")


Saving Modified_Bengali_Review_Dataset.csv to Modified_Bengali_Review_Dataset.csv
✅ Dataset Loaded and Split!


Map:   0%|          | 0/9445 [00:00<?, ? examples/s]

Map:   0%|          | 0/2362 [00:00<?, ? examples/s]

✅ Tokenization Completed!


In [ ]:
# Step 5: Configure LoRA for Fine-Tuning
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1
)

# Apply LoRA to XLM-R
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA Applied Successfully!")

trainable params: 887,042 || all params: 278,932,228 || trainable%: 0.3180
✅ LoRA Applied Successfully!


In [ ]:
# Step 6: Define Training Arguments
training_args = TrainingArguments(
    output_dir="./xlmr_results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    fp16=True,
    report_to="none"
)

print("✅ Training Arguments Set!")

✅ Training Arguments Set!


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# Step 7: Define Evaluation Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary")
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1-score": f1
    }

print("✅ Evaluation Metrics Defined!")

✅ Evaluation Metrics Defined!


In [ ]:
# Step 8: Resume from Checkpoint (if exists)
checkpoint_path = None
if os.path.exists("./xlmr_results") and len(os.listdir("./xlmr_results")) > 0:
    checkpoint_path = "./xlmr_results"
    print(f"🔄 Resuming from last checkpoint: {checkpoint_path}")
else:
    print("🆕 No checkpoint found, starting fresh training!")

🆕 No checkpoint found, starting fresh training!


In [ ]:
# Step 9: Initialize Trainer & Start Training
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# 🚀 Start Training
print("🚀 Training XLM-R with LoRA Started...")
trainer.train(resume_from_checkpoint=checkpoint_path)
print("✅ Training Completed!")


<ipython-input-14-99642e41e283>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


🚀 Training XLM-R with LoRA Started...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1-score
1,0.624000,0.495416,0.892464,0.852649,0.757353,0.802181
2,0.434200,0.413477,0.914056,0.872075,0.822059,0.846329
3,0.018800,0.415955,0.917019,0.874613,0.830882,0.852187


✅ Training Completed!


In [ ]:
# Step 10: Evaluate Model
print("📊 Running Evaluation on Test Set...")
eval_results = trainer.evaluate()
print("✅ Evaluation Completed!")

for key, value in eval_results.items():
    print(f"{key}: {value}")

📊 Running Evaluation on Test Set...


✅ Evaluation Completed!
eval_loss: 0.4134773313999176
eval_accuracy: 0.9140558848433531
eval_precision: 0.8720748829953198
eval_recall: 0.8220588235294117
eval_f1-score: 0.846328538985617
eval_runtime: 12.2711
eval_samples_per_second: 192.484
eval_steps_per_second: 48.162
epoch: 3.0


In [ ]:
# Step 11: Save the Fine-Tuned Model
trainer.save_model("./xlmr_bfrd")
tokenizer.save_pretrained("./xlmr_bfrd")

# Save Evaluation Results
with open("xlmr_results.json", "w") as f:
    json.dump(eval_results, f)

# Zip Model Folder
shutil.make_archive("xlmr_bfrd", 'zip', "./xlmr_bfrd")

# Download Model & Results
files.download("xlmr_bfrd.zip")
files.download("xlmr_results.json")

print("✅ Fine-Tuned Model & Results Saved Successfully!")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Fine-Tuned Model & Results Saved Successfully!
